In [3]:
!pip install ultralytics torch-pruning roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 139.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [4]:
from ultralytics import YOLO

model = YOLO("/content/tavuk_yolo.pt")
model.info()  # Bu komut tüm mimariyi ve parametre özetini yazdırır

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model summary: 130 layers, 3,012,798 parameters, 0 gradients, 8.2 GFLOPs


(130, 3012798, 0, 8.2038272)

In [5]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="uwbLCWSnHV2geuphu3jv")
project = rf.workspace("deneme-nq1dn").project("chicken-invasion-detection-zqoa3")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to chicken-invasion-detection-1 in yolov8:: 100%|██████████| 4362/4362 [00:00<00:00, 10615.72it/s]


In [30]:
import yaml

# nc: 11 (Ship, Chicken, Powerup vb. toplam sınıf sayın)
custom_model_yaml = """
nc: 11
scales:
  n: [0.33, 0.20, 1024]  # 0.15'ten 0.20'ye çıkardık

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[15, 18, 21], 1, Detect, [nc]]
"""

with open('custom_yolo_v2.yaml', 'w') as f:
    f.write(custom_model_yaml)

In [31]:
from ultralytics import YOLO
import os

# 1. Modeli Tanımla ve Eğitimi Başlat
model = YOLO('custom_yolo_v2.yaml')
model.load("tavuk_yolo.pt") # Eski ağırlıklardan transfer öğrenme yap

results = model.train(
    data="/content/chicken-invasion-detection-1/data.yaml",
    epochs=20,
    imgsz=640,
    batch=32,
    lr0=0.01,
    device=0,
    project="Medium_Chicken_YOLO",
    name="width_0_20_final"
)

# --- 3. TEST VE METRİK BASMA ---
print("\n--- 0.20 MODEL TESTİ BAŞLIYOR ---")
best_path = str(results.save_dir) + "/weights/best.pt"
final_model = YOLO(best_path)

# Parametre sayısını en başta görelim
final_model.info()

metrics = final_model.val(
    data="/content/chicken-invasion-detection-1/data.yaml",
    split="test",
    imgsz=640
)

print("\n" + "="*50)
print(f"mAP50 (Doğruluk):           {metrics.box.map50:.4f}")
print(f"Inference Hızı (A100):      {metrics.speed['inference']:.2f} ms")
print("="*50)

# Sınıf bazlı metrikler
for i, name in final_model.names.items():
    if i < len(metrics.box.maps):
        print(f"Sınıf [{name}]: mAP50 -> {metrics.box.maps[i]:.4f}")

WARNING ⚠️ no model scale passed. Assuming scale='n'.
Transferred 121/355 items from pretrained weights
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/chicken-invasion-detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=custom_yolo_v2.yaml, momentum=0.

In [32]:
from ultralytics import YOLO

model = YOLO("/content/budama_sonrası.pt")
model.info()  # Bu komut tüm mimariyi ve parametre özetini yazdırır

custom_YOLO_v2 summary: 130 layers, 2,126,254 parameters, 0 gradients, 6.4 GFLOPs


(130, 2126254, 0, 6.3906688)